# OOP Week 8 -- Strategy Pattern

**Course:** Object-Oriented Programming (Year 2)
**Session:** 3 hours
**Prerequisites:** Weeks 1-7
**Focus:** interchangeable algorithms, config-driven selection

---

## Learning Objectives

1. Explain the Strategy Pattern and when to use it
2. Implement strategies as interchangeable classes
3. Select strategies at runtime based on configuration
4. Compose multiple strategies
5. Recognize Strategy in real-world software

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/YOUR-ORG/YOUR-REPO.git
# %cd YOUR-REPO

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Section 1: What is the Strategy Pattern?

The Strategy Pattern defines a **family of algorithms**, puts each one in a **separate class**, and makes them **interchangeable**.

### The GPS Analogy

Your GPS app has multiple routing strategies:
- **Fastest Route** -- minimize time
- **Shortest Route** -- minimize distance
- **Scenic Route** -- prefer highways with nice views
- **Avoid Tolls** -- find free roads

All strategies have the same interface: they take a start and end point and return a route. You can swap them without changing the rest of the app.

### In Our Pipeline

We have already been using this pattern! Our cleaning strategies (RangeCleaner, MissingCleaner) and analyzer types (MeanAnalyzer, StdAnalyzer) are all strategies. Now we formalize the pattern.

---
## Section 2: Cleaning Strategies

In [ ]:
class CleaningStrategy:
    """Base strategy for cleaning decisions."""
    def should_keep(self, row):
        """Return True to keep the row, False to drop it."""
        return True


class DropMissing(CleaningStrategy):
    """Drop rows with missing values."""
    def __init__(self, columns):
        self.columns = columns

    def should_keep(self, row):
        for col in self.columns:
            val = row.get(col)
            if val is None or (isinstance(val, str) and val.strip() == ""):
                return False
        return True


class DropOutOfRange(CleaningStrategy):
    """Drop rows where a value is outside a range."""
    def __init__(self, column, low, high):
        self.column = column
        self.low = low
        self.high = high

    def should_keep(self, row):
        val = row.get(self.column)
        if isinstance(val, (int, float)):
            return self.low <= val <= self.high
        return True  # non-numeric: keep


class DropDuplicates(CleaningStrategy):
    """Drop rows with duplicate values in a column."""
    def __init__(self, column):
        self.column = column
        self._seen = set()

    def should_keep(self, row):
        val = row.get(self.column)
        if val in self._seen:
            return False
        self._seen.add(val)
        return True


print("Three strategies, same interface: should_keep(row) -> bool")

**Expected Output:**
```
Three strategies, same interface: should_keep(row) -> bool
```

---
## Section 3: The Configurable Cleaner

In [ ]:
class ConfigurableCleaner:
    """Cleaner that uses interchangeable strategies."""

    def __init__(self, strategies=None):
        self.strategies = strategies or []
        self.stats = {"checked": 0, "kept": 0, "dropped": 0}

    def add_strategy(self, strategy):
        """Add a cleaning strategy."""
        self.strategies.append(strategy)
        return self  # for chaining

    def clean(self, data):
        """Apply all strategies. A row must pass ALL to be kept."""
        result = []
        self.stats = {"checked": 0, "kept": 0, "dropped": 0}
        for row in data:
            self.stats["checked"] += 1
            if all(s.should_keep(row) for s in self.strategies):
                result.append(row)
                self.stats["kept"] += 1
            else:
                self.stats["dropped"] += 1
        return result


# Build from config
data = [
    {"id": 1, "value": 25},
    {"id": 2, "value": None},
    {"id": 3, "value": 200},
    {"id": 4, "value": 50},
    {"id": 5, "value": 50},   # duplicate value
]

cleaner = ConfigurableCleaner()
cleaner.add_strategy(DropMissing(["value"]))
cleaner.add_strategy(DropOutOfRange("value", 0, 100))
cleaner.add_strategy(DropDuplicates("value"))

clean = cleaner.clean(data)
print("Results:", clean)
print("Stats:", cleaner.stats)

**Expected Output:**
```
Results: [{'id': 1, 'value': 25}, {'id': 4, 'value': 50}]
Stats: {'checked': 5, 'kept': 2, 'dropped': 3}
```

---
## Section 4: Config-Driven Strategy Selection

The real power: build the entire cleaning pipeline from a configuration dictionary. No code changes needed to add or remove strategies.

In [ ]:
STRATEGY_MAP = {
    "drop_missing": lambda cfg: DropMissing(cfg["columns"]),
    "drop_range": lambda cfg: DropOutOfRange(cfg["column"], cfg["low"], cfg["high"]),
    "drop_duplicates": lambda cfg: DropDuplicates(cfg["column"]),
}

def build_cleaner_from_config(config):
    """Build a ConfigurableCleaner from a config dict."""
    cleaner = ConfigurableCleaner()
    for step in config.get("cleaning_steps", []):
        name = step["strategy"]
        if name not in STRATEGY_MAP:
            raise ValueError("Unknown strategy: " + name)
        strategy = STRATEGY_MAP[name](step)
        cleaner.add_strategy(strategy)
        print("Added strategy: " + name)
    return cleaner


# Config (could come from a YAML/JSON file)
config = {
    "cleaning_steps": [
        {"strategy": "drop_missing", "columns": ["value"]},
        {"strategy": "drop_range", "column": "value", "low": 0, "high": 100},
    ]
}

cleaner = build_cleaner_from_config(config)
result = cleaner.clean(data)
print("Clean:", result)

**Expected Output:**
```
Added strategy: drop_missing
Added strategy: drop_range
Clean: [{'id': 1, 'value': 25}, {'id': 4, 'value': 50}, {'id': 5, 'value': 50}]
```

---
### Try It!

Create a new strategy `DropStatus(CleaningStrategy)` that drops rows where `status` equals a given value (e.g., 'error'). Add it to STRATEGY_MAP and build a cleaner that uses it.

In [ ]:
# YOUR CODE HERE


---
## Mini-Quiz

In [ ]:
# Q1: What is the Strategy Pattern?
# Answer: 

# Q2: What method do all cleaning strategies share?
# Answer: 

# Q3: What advantage does config-driven strategy selection give?
# Answer: 

---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)